1. Objective

This notebook serves as the final integration layer of the Bosnian Heritage Assistant. We combine semantic vector search with a machine learning classifier to solve the "Generic Search Problem."
2. The Hybrid Architecture

The system operates on a Three-Layer Logic:

    Retrieval Layer (ChromaDB): Uses semantic embeddings to find the top 10 sites that match the user's natural language query (e.g., "I want to see old castles").

    Inference Layer (Random Forest): Takes the 10 candidates from ChromaDB and re-ranks them based on the Inference Logic trained in Notebook 02 (Region, Season, Interest compatibility).

    Heuristic Layer (Popularity Toggle): Adjusts the final scores to either favor "Must-See" landmarks or "Hidden Gems" based on intent detection (keywords like quiet, secret, mir, tišina).

3. Key Features

    Algorithm Battle: Comparison between raw vector search results and AI-ranked results.

    Cultural Context: Integration of the regional_food_heritage dictionary for gastronomic mapping.

    Gradio UI: A functional web interface for real-time interaction and testing.

In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. SEARCH FOR EXISTING DATABASE PATHS
possible_paths = [
    "/content/drive/MyDrive/Y2S1/MLSTProject/heritage_db",
    "/content/heritage_db",
    "./heritage_db",
    "../heritage_db"
]

db_path = None
for path in possible_paths:
    if os.path.exists(path):
        db_path = path
        break

# If no path exists, default to local directory
if not db_path:
    db_path = "./heritage_db"

print(f"🔗 Connecting ChromaDB to path: {db_path}")

# 2. INITIALIZE CLIENT
chroma_client = chromadb.PersistentClient(path=db_path)

# 3. USE get_or_create_collection TO PREVENT CRASHING
collection = chroma_client.get_or_create_collection(name="bih_heritage")

# 4. AUTO-SEED IF COLLECTION IS EMPTY (Fixes missing collection issue)
if collection.count() == 0:
    print("⚠️ Collection empty/missing. Auto-populating from CSV...")
    import pandas as pd

    csv_paths = [
        "/content/drive/MyDrive/Y2S1/MLSTProject/BiH_Heritage_Final_Clean.csv",
        "BiH_Heritage_Final_Clean.csv"
    ]

    csv_found = None
    for cp in csv_paths:
        if os.path.exists(cp):
            csv_found = cp
            break

    if csv_found:
        df = pd.read_csv(csv_found)
        df = df.fillna({'region': 'Central Bosnia', 'category': 'nature', 'historical_period': 'Modern'})

        ids = [str(row['name']) for _, row in df.iterrows()]
        documents = [
            f"{row['name']} in {row['location']} {row['region']}. {row['description']}"
            for _, row in df.iterrows()
        ]
        metadatas = [
            {
                "location": str(row['location']),
                "region": str(row['region']),
                "category": str(row['category'])
            }
            for _, row in df.iterrows()
        ]

        collection.upsert(ids=ids, documents=documents, metadatas=metadatas)
        print(f"✅ Auto-seeded {collection.count()} items into ChromaDB!")
    else:
        print("❌ CSV file not found to seed ChromaDB.")

🔗 Connecting ChromaDB to path: /content/drive/MyDrive/Y2S1/MLSTProject/heritage_db


In [ ]:
import joblib
import pandas as pd
import os
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Paths (Using your V2 brain which is a dictionary)
BASE_PATH = '/content/drive/MyDrive/Y2/Y2S1/MLSTProject/'
model_path = os.path.join(BASE_PATH, 'heritage_brain_v2.pkl')

# 3. Unpack the "Project Bundle"
bundle = joblib.load(model_path)

# Extracting everything into variables
clf = bundle['model']
le_interest = bundle['le_interest']
le_season = bundle['le_season']
le_region = bundle['le_region']
le_category = bundle['le_category']
le_period = bundle['le_period']
le_pop = bundle['le_pop']

print("✅ SUCCESS: Hybrid System Online.")
print(f"Features expected by the model: {clf.feature_names_in_}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ SUCCESS: Hybrid System Online.
Features expected by the model: ['user_interest' 'current_season' 'user_region' 'limited_time'
 'site_region' 'site_category' 'site_pop' 'period']


In [ ]:
!pip install -q chromadb

In [26]:
import chromadb

# 1. Connect to where your data is stored
# Make sure this path matches where you saved your DB earlier!
client = chromadb.PersistentClient(path="/content/drive/MyDrive/Y2/Y2S1/MLSTProject/heritage_db")

# 2. Get your collection (the name must match what you used during creation)
# If you don't remember the name, run: print(client.list_collections())
collection = client.get_collection(name="bih_heritage")

print("✅ Collection connected successfully!")

✅ Collection connected successfully!


In [27]:
def prepare_competitive_features(df):
    df_enc = df.copy()

    # Matching the exact names from your 'features_expected_by_the_model' list
    df_enc['user_interest'] = le_interest.transform(df_enc['user_interest'])
    df_enc['current_season'] = le_season.transform(df_enc['current_season'])
    df_enc['user_region'] = le_region.transform(df_enc['user_region_pref']) # Map pref to user_region

    df_enc['site_region'] = le_region.transform(df_enc['site_region'])
    df_enc['site_category'] = le_category.transform(df_enc['site_category'])
    df_enc['period'] = le_period.transform(df_enc['historical_period']) # Map historical_period to period

    if hasattr(le_pop, 'transform'):
        df_enc['site_pop'] = le_pop.transform(df_enc['site_popularity'])
    else:
        df_enc['site_pop'] = df_enc['site_popularity'].astype(int)

    # Drop the old names so the model doesn't get confused
    return df_enc[['user_interest', 'current_season', 'user_region', 'limited_time',
                  'site_region', 'site_category', 'site_pop', 'period']]

#Phase 5: The Smart Recommendation Function

This brings the Librarian (Vector) and the Expert (ML Model) together

In [28]:
def elite_hybrid_search(user_query, user_reg_pref, user_int, season, limited_time=False):
    # 1. RETRIEVAL (ChromaDB) - Find top 50 candidates
    results = collection.query(query_texts=[user_query], n_results=50)
    metadatas = results['metadatas'][0]
    ids = results['ids'][0]
    query_lower = user_query.lower().strip()

    # Pre-detect broad intents
    is_food = any(w in query_lower for w in ['eat', 'food', 'hungry', 'cevapi', 'pita', 'fast food'])
    is_hike = any(w in query_lower for w in ['hike', 'nature', 'mountain', 'walk', 'climb'])

    scored_list = []

    for i in range(len(metadatas)):
        meta = metadatas[i]

        # 2. MODEL INFERENCE (The Random Forest Expert)
        feat_df = pd.DataFrame([{
            'user_interest': user_int,
            'current_season': season,
            'user_region_pref': user_reg_pref,
            'limited_time': 1 if limited_time else 0,
            'site_region': meta['region'],
            'site_category': meta['category'],
            'historical_period': meta['historical_period'],
            'site_popularity': meta['popularity_level']
        }])

        X_test = prepare_competitive_features(feat_df)
        # Probability that this is a "Good Match" (0 to 100)
        base_score = clf.predict_proba(X_test)[0][1] * 100

        # 3. HEURISTIC RE-RANKING (The Precision Layer)
        site_cat = str(meta.get('category', '')).lower().strip()
        site_loc = str(meta.get('location', '')).lower().strip()
        site_reg = str(meta.get('region', '')).lower().strip()

        # A. Dynamic Geographic Boost
        # If the user literally typed the city name or region name
        if site_loc in query_lower or site_reg in query_lower:
            base_score += 100
        elif user_reg_pref.lower() == site_reg:
            base_score += 20 # Convenience boost

        # B. Category & Synonym Boost
        # Mapping common words to your dataset categories
        synonyms = {
            'fortress': ['castle', 'tvrđava', 'grad', 'fort'],
            'religious': ['mosque', 'church', 'monastery', 'džamija', 'crkva'],
            'nature': ['park', 'waterfall', 'river', 'mountain', 'lake']
        }

        # Check for direct match or synonym match
        if site_cat in query_lower:
            base_score += 75
        else:
            for official_cat, words in synonyms.items():
                if site_cat == official_cat and any(w in query_lower for w in words):
                    base_score += 75
                    break

        # C. Intent Safeguards
        if is_food:
            if site_cat in ['traditional village', 'urban heritage']: base_score += 40
            if site_cat in ['nature', 'monument']: base_score -= 30

        if is_hike and site_cat == 'nature':
            base_score += 40

        scored_list.append({
            'name': ids[i],
            'score': round(base_score, 2),
            'category': meta['category'],
            'region': meta['region'],
            'city': meta.get('location', 'Unknown')
        })

    # Sort results: Best Match first
    return sorted(scored_list, key=lambda x: x['score'], reverse=True)

#Phase 6: Interactive Prototype

Finally, the chat interface for the user.

In [29]:
# Try a natural language query
query = "I want to visit an old fortress near Zenica but I want a hidden gem"

# Search the collection
results = collection.query(
    query_texts=[query],
    n_results=3
)

# Print the top matches
print("--- TOP 3 CANDIDATES ---")
for i, doc in enumerate(results['documents'][0]):
    print(f"\nMatch {i+1}: {doc}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- TOP 3 CANDIDATES ---

Match 1: An 16th-century architectural gem known for its unique stone carvings and history. mosque, architecture, history, Čajniče

Match 2: A popular mountain excursion site located 8km from Zenica. It offers fresh air, walking trails, and a small ski lift for beginners, making it a favorite for local winter outings. snow, sledding, hiking, Zenica, fresh air, monument, winter

Match 3: Not the fortress, but the ruins of the medieval silver mining town that once rivaled Dubrovnik. silver, mining, ruins, history


In [30]:
# 1. Connect to the confirmed collection
collection = client.get_collection(name="bih_heritage")

def elite_hybrid_search(user_query, user_reg_pref, user_int, season, limited_time=False):
    # STEP A: RETRIEVAL
    results = collection.query(query_texts=[user_query], n_results=10)
    metadatas = results['metadatas'][0]
    ids = results['ids'][0]

    pop_map = {'A': 3, 'B': 2, 'C': 1}
    scored_list = []

    # STEP B: INFERENCE
    for i in range(len(metadatas)):
        meta = metadatas[i]

        # 1. Safely parse and fall back for metadata
        raw_pop = str(meta.get('popularity_level', 'B')).strip().upper()
        clean_pop = pop_map.get(raw_pop, 2)

        site_reg = str(meta.get('region', le_region.classes_[0])).title()
        if site_reg not in le_region.classes_:
            site_reg = le_region.classes_[0]

        raw_cat = str(meta.get('category', 'building')).lower().strip()
        clean_cat = le_category.classes_[0]
        for known in le_category.classes_:
            if known in raw_cat:
                clean_cat = known
                break

        site_period = str(meta.get('historical_period', le_period.classes_[0])).strip()
        if site_period not in le_period.classes_:
            site_period = le_period.classes_[0]

        # 2. Build feature frame
        feat_df = pd.DataFrame([{
            'user_interest': user_int,
            'current_season': season,
            'user_region_pref': user_reg_pref,
            'limited_time': 1 if limited_time else 0,
            'site_region': site_reg,
            'site_category': clean_cat,
            'historical_period': site_period,
            'site_popularity': clean_pop
        }])

        # 3. Predict match probability
        X_test = prepare_competitive_features(feat_df)
        base_score = clf.predict_proba(X_test)[0][1] * 100

        # STEP C: PRECISION & HEURISTICS
        site_loc = str(meta.get('location', '')).lower()
        if site_loc and site_loc in user_query.lower():
            base_score += 100

        # Include category and region in return payload for clean UI rendering
        scored_list.append({
            'name': ids[i],
            'score': round(base_score, 2),
            'city': meta.get('location', 'Unknown'),
            'category': clean_cat,
            'region': site_reg
        })

    # Sort by the final AI Score
    return sorted(scored_list, key=lambda x: x['score'], reverse=True)

# --- TEST IT OUT ---
test_res = elite_hybrid_search("I want a fortress near Zenica", "Central Bosnia", "History", "Summer")
print(f"🤖 TOP MATCH: {test_res[0]['name']} ({test_res[0]['score']}% Match)")

🤖 TOP MATCH: Vranduk Fortress (190.78% Match)


In [31]:
# 1. Get the Ranked Results from your Hybrid Engine
query = "I want to visit an old fortress near Zenica but I want a hidden gem"
ranked_results = elite_hybrid_search(query, "Central Bosnia", "History", "Summer")

print(f"🤖 TRAVEL ASSISTANT RESPONSE for: '{query}'\n")

if not ranked_results:
    print("I'm sorry, I couldn't find anything matching that specific request.")
else:
    print(f"I found some great spots for you! Based on your interest in {ranked_results[0]['category']}, here are the top picks:")
    print("-" * 50)

    for i, res in enumerate(ranked_results[:3]):
        # We use 'name' from our scored list, which we got from ChromaDB IDs
        print(f"{i+1}. 🏰 {res['name']} ({res['score']}% AI Match)")
        print(f"   📍 Location: {res['region']}")
        print(f"   📂 Category: {res['category']}")
        print("-" * 30)

    print("\n💡 Pro-tip: Since you're looking for fortresses, Vranduk is the most iconic one near Zenica!")

🤖 TRAVEL ASSISTANT RESPONSE for: 'I want to visit an old fortress near Zenica but I want a hidden gem'

I found some great spots for you! Based on your interest in nature, here are the top picks:
--------------------------------------------------
1. 🏰 Smetovi Zenica (142.32% AI Match)
   📍 Location: Central Bosnia
   📂 Category: nature
------------------------------
2. 🏰 Tvrđava Soko (Šipovo) (87.8% AI Match)
   📍 Location: Central Bosnia
   📂 Category: fortress
------------------------------
3. 🏰 Fortress of Glamoč (85.25% AI Match)
   📍 Location: Central Bosnia
   📂 Category: fortress
------------------------------

💡 Pro-tip: Since you're looking for fortresses, Vranduk is the most iconic one near Zenica!


In [32]:
import joblib

# Bundle the model and encoders from your notebook memory
brain_package = {
    'model': clf,
    'le_interest': le_interest,
    'le_season': le_season,
    'le_region': le_region,
    'le_category': le_category,
    'le_period': le_period,
    'le_pop': le_pop
}

# Save it locally where app.py looks for it
joblib.dump(brain_package, 'heritage_brain_v2.pkl')
print("✅ Saved heritage_brain_v2.pkl to local directory!")

✅ Saved heritage_brain_v2.pkl to local directory!


#finally, the ui


In [33]:
!pip install -q streamlit
!npm install -q -g localtunnel
!pip install -q groq

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 3s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.1 MB/s eta 0:00:00


In [34]:
import os
from google.colab import drive

# Mount Google Drive if not mounted
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

print("🔍 Searching for required assets...")

brain_found = False
for root, dirs, files in os.walk("/content"):
    if "heritage_brain_v2.pkl" in files:
        print(f"✅ FOUND MODEL AT: {os.path.join(root, 'heritage_brain_v2.pkl')}")
        brain_found = True
        break

if not brain_found:
    print("❌ heritage_brain_v2.pkl missing! Upload it to /content or Google Drive.")

db_found = False
for root, dirs, files in os.walk("/content"):
    if "heritage_db" in dirs:
        print(f"✅ FOUND VECTOR DB AT: {os.path.join(root, 'heritage_db')}")
        db_found = True
        break

if not db_found:
    print("❌ heritage_db folder missing! Upload it to /content or Google Drive.")

🔍 Searching for required assets...
✅ FOUND MODEL AT: /content/heritage_brain_v2.pkl
✅ FOUND VECTOR DB AT: /content/drive/MyDrive/Y2/Y2S1/MLSTProject/heritage_db


In [116]:
%%writefile backend.py
import os
import re
import chromadb
from chromadb.utils import embedding_functions
from groq import Groq

class KulturAIBackend:
    def __init__(self, model_path="heritage_brain_v2.pkl", db_path="heritage_db"):
        possible_db_paths = [
            db_path,
            "/content/drive/MyDrive/Y2S1/MLSTProject/heritage_db",
            "/content/heritage_db",
            "../heritage_db"
        ]
        actual_db_path = next((p for p in possible_db_paths if os.path.exists(p)), "heritage_db")

        ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
        self.chroma_client = chromadb.PersistentClient(path=actual_db_path)

        try:
            self.collection = self.chroma_client.get_collection(name="bih_heritage", embedding_function=ef)
        except Exception:
            self.collection = self.chroma_client.get_or_create_collection(name="bih_heritage", embedding_function=ef)

        self.candidate_models = [
            "llama-3.3-70b-versatile",
            "llama3-8b-8192",
            "mixtral-8x7b-32768"
        ]

    def _get_active_chat_models(self, client: Groq) -> list:
        try:
            available = client.models.list()
            active_ids = [m.id for m in available.data]
            chat_models = [
                m for m in active_ids
                if not any(excluded in m.lower() for excluded in ["whisper", "orpheus", "tts", "stt", "embed", "vision"])
            ]
            ordered = [m for m in self.candidate_models if m in chat_models]
            ordered.extend([m for m in chat_models if m not in ordered])
            return ordered if ordered else self.candidate_models
        except Exception:
            return self.candidate_models

    def is_casual_greeting(self, text: str) -> bool:
        greetings = {"hi", "hello", "hey", "zdravo", "kako si", "test", "how are you"}
        cleaned = re.sub(r'[^\w\s]', '', text.strip().lower())
        return cleaned in greetings or (len(cleaned.split()) <= 1 and cleaned not in ["olovo", "kladanj", "jajce", "mostar", "tuzla", "sarajevo"])

    def sanitize_query(self, query: str) -> str:
        # Strips trailing underscores, typos, or symbols (e.g. "olovo_" -> "olovo")
        cleaned = re.sub(r'[_\W]+', ' ', query).strip()
        return cleaned if cleaned else query

    def _clean_thinking_tags(self, text: str) -> str:
        # Removes internal model reasoning like <think>...</think>
        cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
        return cleaned.strip()

    def extract_location_anchor(self, chat_history: list) -> str:
        """Scans recent conversation history to extract persistent town/location context."""
        if not chat_history:
            return ""

        # Priority cities to lock context onto if mentioned previously
        known_locations = ["olovo", "kladanj", "jajce", "mostar", "tuzla", "sarajevo", "banja luka", "bihac", "trebinje", "konjic", "travnik", "visoko"]

        for msg in reversed(chat_history):
            if msg["role"] == "user":
                clean_msg = self.sanitize_query(msg["content"]).lower()
                for loc in known_locations:
                    if loc in clean_msg:
                        return loc
        return ""

    def hybrid_recommend(self, query: str, profile: dict, chat_history: list = None) -> list:
        clean_q = self.sanitize_query(query)
        if self.is_casual_greeting(clean_q):
            return []

        try:
            # Check for persistent location context in history
            location_anchor = self.extract_location_anchor(chat_history or [])

            # Formulate targeted query
            if location_anchor and location_anchor not in clean_q.lower():
                search_query = f"{location_anchor} {clean_q}"
            else:
                search_query = clean_q

            results = self.collection.query(query_texts=[search_query], n_results=8)
            metadatas = results['metadatas'][0] if results.get('metadatas') else []
            ids = results['ids'][0] if results.get('ids') else []
            distances = results['distances'][0] if results.get('distances') else [0.5] * len(ids)

            scored_sites = []
            for i, meta in enumerate(metadatas):
                raw_dist = distances[i]
                conf_score = round(max(0, (1 - (raw_dist / 2)) * 100), 1)

                scored_sites.append({
                    'name': ids[i],
                    'score': conf_score,
                    'location': meta.get('location', 'BiH'),
                    'region': meta.get('region', 'Central Bosnia'),
                    'category': meta.get('category', 'leisure'),
                    'desc': meta.get('description', 'No detailed description available.')
                })

            # Hard Keyword Fallback: Ensure matching records for location anchor are forced in
            target_keyword = location_anchor or clean_q.lower()
            all_docs = self.collection.get()
            matched_fallback = []

            for i, meta in enumerate(all_docs.get('metadatas', [])):
                loc = str(meta.get('location', '')).lower()
                desc = str(meta.get('description', '')).lower()
                doc_id = all_docs['ids'][i]

                if target_keyword and (target_keyword in loc or target_keyword in desc):
                    # Prevent duplicates
                    if not any(s['name'] == doc_id for s in scored_sites):
                        matched_fallback.append({
                            'name': doc_id,
                            'score': 90.0,
                            'location': meta.get('location', 'BiH'),
                            'region': meta.get('region', 'Central Bosnia'),
                            'category': meta.get('category', 'leisure'),
                            'desc': meta.get('description', '')
                        })

            # Prioritize fallback matches if a specific location was targeted
            if matched_fallback:
                scored_sites = matched_fallback + scored_sites

            return sorted(scored_sites, key=lambda x: x['score'], reverse=True)
        except Exception as e:
            print(f"[VECTOR DB ERROR]: {e}")
            return []

    def generate_llm_response(self, prompt: str, history: list = None, profile: dict = None, context_matches: list = None, api_key: str = None) -> tuple:
        key_to_use = (
            api_key
            or os.getenv("GROQ_API_KEY")
            or "YOUR_GROQ_API_KEY"
        )

        try:
            client = Groq(api_key=key_to_use)
            active_models = self._get_active_chat_models(client)
        except Exception as e:
            return f"Client Initialization Error: {e}", "Initialization failed."

        profile = profile or {}
        profile_str = f"Interest: {profile.get('interest', 'Any')}, Season: {profile.get('season', 'Any')}, Region: {profile.get('region', 'Any')}"

        sys_msg = (
            "You are KulturAI, a precision travel assistant for Bosnia & Herzegovina.\n\n"
            f"USER PROFILE:\n{profile_str}\n\n"
            "CRITICAL RULES:\n"
            "1. STRICT GROUNDING: ONLY recommend destinations from the VERIFIED KNOWLEDGE BASE CONTEXT. Do NOT invent fake places.\n"
            "2. NO TRAVEL ESTIMATES: DO NOT invent numeric drive times or walking times.\n"
            "3. NO REPETITION: State information once concisely. Do not repeat lists or paragraphs.\n"
            "4. CONTEXT CONTINUITY: If the user mentions having a car or changing modes of transit, adapt your advice to the previously discussed location and context matches.\n"
            "5. OUTPUT FORMAT: Respond directly to the user. Do not include internal thinking tags (<think>) or chain-of-thought blocks."
            "6. LANGUAUGE:  Use English unless asked in another lamguage."
        )

        if context_matches:
            db_context = "\n\nVERIFIED KNOWLEDGE BASE CONTEXT:\n" + "\n".join(
                [f"- {m['name']} (Town/Location: {m['location']}): {m['desc']}" for m in context_matches[:6]]
            )
            sys_msg += db_context
        else:
            sys_msg += "\n\n(No database context available. State clearly that you don't have records for this location in your verified database instead of inventing attractions.)"

        messages = [{"role": "system", "content": sys_msg}]

        if history:
            for m in history[-6:]:
                if m["role"] in ["user", "assistant"]:
                    clean_content = self._clean_thinking_tags(m["content"]) if m["role"] == "assistant" else m["content"]
                    messages.append({"role": m["role"], "content": clean_content})

        messages.append({"role": "user", "content": prompt})

        last_err = ""
        for model_id in active_models:
            try:
                res = client.chat.completions.create(
                    messages=messages,
                    model=model_id,
                    temperature=0.0,
                    max_tokens=600
                )

                raw_response = res.choices[0].message.content
                clean_response = self._clean_thinking_tags(raw_response)

                rationale = (
                    f"**Engine**: `{model_id}` | "
                    f"**Context Matches**: {len(context_matches) if context_matches else 0} | "
                    f"**Injected Context**: {', '.join([m['name'] for m in context_matches[:3]]) if context_matches else 'None'}"
                )

                return clean_response, rationale
            except Exception as e:
                last_err = f"{model_id}: {str(e)}"
                continue

        return f"Groq Generation Error: {last_err}", "Execution failed."

Overwriting backend.py


In [36]:
!pip install -q streamlit chromadb sentence-transformers joblib
!wget -q -O cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb

(Reading database ... 122801 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.9.1) over (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [110]:
%%writefile app.py
import streamlit as st
import os
from backend import KulturAIBackend

st.set_page_config(page_title="KulturAI Concierge", layout="wide", page_icon="🇧🇦")

@st.cache_resource
def get_backend():
    return KulturAIBackend()

try:
    backend = get_backend()
except Exception as e:
    st.error(f"Backend Initialization Error: {e}")
    st.stop()

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Zdravo! How can I help you explore Bosnia & Herzegovina today?"}
    ]

if "last_matches" not in st.session_state:
    st.session_state.last_matches = []

# --- Sidebar ---
with st.sidebar:
    st.header("🎛️ Traveler Profile")
    interest = st.selectbox("Primary Interest", ["All / Not Selected", "History", "Nature", "Culture", "Food", "Architecture"])
    season = st.selectbox("Season", ["All / Not Selected", "Summer", "Winter", "Spring", "Autumn"])
    region = st.selectbox("Preferred Region", ["All / Not Selected", "Sarajevo Region", "Herzegovina", "Central Bosnia", "Northern Bosnia"])
    express = st.checkbox("Express Trip (<1.5 hrs)")

    profile = {
        "interest": interest if interest != "All / Not Selected" else "Any",
        "season": season if season != "All / Not Selected" else "Any",
        "region": region if region != "All / Not Selected" else "Any",
        "express": express
    }

    st.markdown("---")
    if st.button("📌 Recommend Based on Profile", use_container_width=True):
        user_prompt = "Can you give me destination recommendations based on my active traveler profile?"
        st.session_state.messages.append({"role": "user", "content": user_prompt})

        matches = backend.hybrid_recommend(user_prompt, profile, st.session_state.messages)
        st.session_state.last_matches = matches

        response, rationale = backend.generate_llm_response(
            prompt=user_prompt,
            history=st.session_state.messages,
            profile=profile,
            context_matches=matches
        )
        st.session_state.messages.append({"role": "assistant", "content": response, "rationale": rationale})
        st.rerun()

    if st.button("🗑️ Clear Chat", use_container_width=True):
        st.session_state.messages = [
            {"role": "assistant", "content": "Zdravo! How can I help you explore Bosnia & Herzegovina today?"}
        ]
        st.session_state.last_matches = []
        st.rerun()

    st.markdown("---")
    with st.expander("📊 Active Database Context", expanded=True):
        if st.session_state.last_matches:
            for match in st.session_state.last_matches[:4]:
                st.markdown(f"**{match['name']}** (`{match['location']}`)")
                st.caption(f"Match: **{match['score']}%** | {match['desc'][:90]}...")
                st.markdown("---")
        else:
            st.caption("No active database context for this query.")

# --- Main Interface ---
st.title("🇧🇦 KulturAI Concierge")
st.caption("Grounded AI Travel Advisor for Bosnia & Herzegovina")

# Render message history once cleanly
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if "rationale" in msg and msg["rationale"]:
            with st.expander("🧠 View AI Concierge Rationale"):
                st.markdown(msg["rationale"])

# Chat Input Handler
if user_input := st.chat_input("Ask about destinations, rivers, or day trips..."):
    # Append User Message
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)

    # Generate Assistant Response
    with st.chat_message("assistant"):
        with st.spinner("Processing travel context..."):
            matches = backend.hybrid_recommend(user_input, profile, st.session_state.messages)
            st.session_state.last_matches = matches

            response, rationale = backend.generate_llm_response(
                prompt=user_input,
                history=st.session_state.messages,
                profile=profile,
                context_matches=matches
            )

            st.markdown(response)
            with st.expander("🧠 View AI Concierge Rationale"):
                st.markdown(rationale)

            st.session_state.messages.append({"role": "assistant", "content": response, "rationale": rationale})
            st.rerun()

Overwriting app.py


In [38]:
#!pip install --force-reinstall "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1"

In [41]:
!pip install -q pyngrok


In [98]:
import os, subprocess, time
from pyngrok import ngrok

# 1. POSTAVI SVOJ GROQ API KLJUČ OVDJE (MORA POČINJATI SA gsk_):
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "YOUR_GROQ_API_KEY")

# 2. Zaustavi sve prethodne pokrenute procese da ne prave konfuziju
!pkill -f streamlit
!pkill -f ngrok

# 3. Postavi ngrok token
NGROK_TOKEN = os.getenv("NGROK_AUTH_TOKEN", "YOUR_NGROK_AUTH_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

# 4. Pokreni Streamlit u pozadini
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(4)

# 5. Generiši stabilan ngrok link
public_url = ngrok.connect(8501)
print("\n" + "="*50)
print(f"🚀 TVOJA APLIKACIJA JE RADI I DUPLE SPREMNA!")
print(f"👉 KLIKNI OVDJE: {public_url}")
print("="*50)


🚀 TVOJA APLIKACIJA JE RADI I DUPLE SPREMNA!
👉 KLIKNI OVDJE: NgrokTunnel: "https://cascade-overvalue-tabasco.ngrok-free.dev" -> "http://localhost:8501"
